# How to think about metrics

_Metrics & Evaluation — measuring models, data & statistics_

**A metric is one number that scores how good a prediction is. Pick it from the cost of being wrong.**

A metric is a single number that scores how good something is — a prediction, a whole model, or a dataset. Higher (or lower) is better, depending on the metric.

       It is the answer to "how well did we do?", boiled down to one comparable number so you can rank models and track progress.

---

This notebook is a practice scaffold for the **How to think about metrics** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, classification_report

# positive class = malignant tumor (the costly miss).
# sklearn's target is 0 = malignant, 1 = benign, so we flip it.
data = load_breast_cancer()
X = data.data
y = (data.target == 0).astype(int)          # 1 = malignant (positive)

# 1) THE SPLIT: never judge a model on the data it learned from.
#    stratify=y keeps the class balance the same in train and test.
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.4, random_state=0, stratify=y)

# 2) fit a simple logistic-regression baseline
clf = make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=5000)).fit(X_tr, y_tr)
y_pred = clf.predict(X_te)                   # uses the default 0.5 threshold

# 3) THE CONFUSION MATRIX: the root of every classification metric.
#    labels=[0,1] -> rows/cols are [negative, positive]
tn, fp, fn, tp = confusion_matrix(y_te, y_pred, labels=[0, 1]).ravel()
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")

# 4) accuracy / precision / recall / F1, all derived from those four counts
print(classification_report(
    y_te, y_pred, target_names=["benign", "malignant"], digits=3))

# the metrics by hand, to show they ARE just fractions of the four boxes:
accuracy  = (tp + tn) / (tp + fp + fn + tn)
precision = tp / (tp + fp)
recall    = tp / (tp + fn)
print(f"accuracy={accuracy:.3f}  precision={precision:.3f}  recall={recall:.3f}")

## Visualize the data & results

_What does a real confusion matrix look like, and which of the four boxes (TP/FP/FN/TN) carry the counts?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix

# positive class = malignant (sklearn target: 0=malignant, 1=benign)
data = load_breast_cancer()
X = data.data
y = (data.target == 0).astype(int)          # 1 = malignant

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.4, random_state=0, stratify=y)

clf = make_pipeline(StandardScaler(),
                    LogisticRegression(max_iter=5000)).fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

# rows = actual [neg, pos], cols = predicted [neg, pos]
cm = confusion_matrix(y_te, y_pred, labels=[0, 1])
print(cm)                                    # [[141, 2], [7, 78]]
tn, fp, fn, tp = cm.ravel()
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print("accuracy :", round((tp + tn) / cm.sum(), 3))
print("recall   :", round(tp / (tp + fn), 3))
print("precision:", round(tp / (tp + fp), 3))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A colleague proudly reports "our churn model is 96% accurate." Only 4% of customers actually churn. Why is this number almost meaningless, and what would you ask for instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find the base rate. — _Only 4% churn, so 96% are non-churners. A model that predicts "no churn" for everyone is right 96% of the time and catches zero churners._
- See that accuracy is gamed by imbalance. — _The 96% comes from the easy majority class, not from any real skill at finding churners — the thing you actually care about._
- Ask for the metric tied to the goal. — _Request the confusion matrix, plus precision and recall on the churn class (or PR-AUC), so you can see how many real churners are caught._

**Answer:** The 96% is just the base rate of non-churners — a do-nothing model that always says "no churn" scores the same. Accuracy is meaningless on imbalanced data. Ask for the confusion matrix and the precision and recall on the churn class (or PR-AUC): those reveal how many actual churners the model catches versus misses.

</details>

**Problem 2.** What is the difference between a loss and a metric, and why can't you usually just train the model directly on the metric you care about (say, F1)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Define each role. — _A loss is what the model trains on; a metric is what you report. They answer different questions._
- Recall what training needs. — _Gradient descent follows the slope of the loss downhill, so the loss must be smooth and differentiable._
- See why F1 fails as a training target. — _F1 (and accuracy) come from counting discrete correct/wrong calls — they are flat then jump, with no useful slope to descend._

**Answer:** A loss is the smooth, optimizable number the model is trained to minimize; a metric is the human-meaningful number you report. You usually cannot train on F1 directly because it is built from discrete counts and has no useful gradient — it is flat then jumps, so gradient descent has nothing to follow. Instead you train on a smooth surrogate loss (like log loss) chosen so that lowering it tends to raise the metric, then report the metric separately.

</details>

**Problem 3.** You are evaluating a model and you move the decision threshold from 0.5 down to 0.2. In terms of TP, FP, FN, TN, what generally happens, and which kinds of problems make a low threshold the right choice?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Track what a lower threshold does. — _A lower cutoff means the model says "yes" more often, so more cases land in the positive column._
- Move the counts. — _More "yes" calls means more true catches (TP up) and fewer misses (FN down), but also more false alarms (FP up) and fewer correct all-clears (TN down)._
- Tie it to cost. — _Lowering the threshold is right when a miss (FN) is far costlier than a false alarm (FP) — you would rather over-alert than let a real case slip._

**Answer:** Lowering the threshold makes the model predict "yes" more readily: TP rises and FN falls (more catches, fewer misses), while FP rises and TN falls (more false alarms). That trade is the right one when a miss is much more expensive than a false alarm — cancer screening, fraud you must not let through, safety alerts — where catching the real cases matters more than the extra false positives.

</details>